# Convert Qwen3-4B-Base into a reasoning model via GRPO by using OpenR1's Math dataset

In [1]:
# Python >= 3.10
# pip install unsloth vllm

from unsloth import FastLanguageModel
from datasets import Dataset, load_dataset
from trl import SFTTrainer, SFTConfig
from transformers import TextStreamer
import torch
import pandas as pd
import numpy as np
import gc

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO 08-21 09:23:03 [__init__.py:241] Automatically detected platform cuda.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"[INFO] Using {device} device")

[INFO] Using cuda device


## (1) Prepare model

In [3]:
max_seq_length = 2048  # increase for longer reasoning traces
lora_rank = 32         # [8, 16, 32, 64, 128] Larger rank => smarter & slower 

In [4]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-Base",
    max_seq_length = max_seq_length,
    load_in_4bit = False,  # False for LoRA 16bit
    fast_inference = True, # Enable vLLM fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.7, # Reduce if out of memory
)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Unsloth: Patching vLLM v1 graph capture
Unsloth: Patching vLLM v0 graph capture
==((====))==  Unsloth 2025.8.9: Fast Qwen3 patching. Transformers: 4.55.2. vLLM: 0.10.1.
   \\   /|    NVIDIA GeForce RTX 4070 Ti SUPER. Num GPUs = 1. Max memory: 15.547 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.1+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.3.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.31. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/Qwen3-4B-Base with actual GPU utilization = 66.38%
Unsloth: Your GPU has CUDA compute capability 8.9 with VRAM = 15.55 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 2048. Num Sequences = 160.
Unsloth: vLLM's KV Cache can use up to 3.37 GB. Also swap space = 6 GB.
Unsloth: Not an error, but `device` is not supported in vLLM. Skipping.
INFO 08-21 09:23:09 [utils.py:326] non-defaul

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


INFO 08-21 09:23:39 [default_loader.py:262] Loading weights took 20.17 seconds
INFO 08-21 09:23:39 [punica_selector.py:19] Using PunicaWrapperGPU.
INFO 08-21 09:23:39 [gpu_model_runner.py:2007] Model loading took 7.6334 GiB and 21.428472 seconds
INFO 08-21 09:23:47 [backends.py:548] Using cache directory: /home/huo/.cache/vllm/torch_compile_cache/9c86beb8af/rank_0_0/backbone for vLLM's torch.compile
INFO 08-21 09:23:47 [backends.py:559] Dynamo bytecode transform time: 7.57 s
INFO 08-21 09:23:53 [backends.py:161] Directly load the compiled graph(s) for dynamic shape from the cache, took 5.123 s
INFO 08-21 09:23:55 [monitor.py:34] torch.compile takes 7.57 s in total
INFO 08-21 09:23:55 [gpu_worker.py:276] Available KV cache memory: 1.91 GiB
INFO 08-21 09:23:56 [kv_cache_utils.py:849] GPU KV cache size: 13,904 tokens
INFO 08-21 09:23:56 [kv_cache_utils.py:853] Maximum concurrency for 2,048 tokens per request: 6.79x
INFO 08-21 09:23:56 [vllm_utils.py:643] Unsloth: Running patched vLLM v1 `

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 43/43 [00:06<00:00,  6.89it/s]

INFO 08-21 09:24:02 [gpu_model_runner.py:2708] Graph capturing finished in 6 secs, took 0.54 GiB
INFO 08-21 09:24:02 [vllm_utils.py:650] Unsloth: Patched vLLM v1 graph capture finished in 6 secs.


INFO 08-21 09:24:02 [core.py:214] init engine (profile, create kv cache, warmup model) took 23.14 seconds
INFO 08-21 09:24:03 [llm.py:298] Supported_tasks: ('generate',)
Unsloth: Just some info: will skip parsing ['pre_feedforward_layernorm', 'post_feedforward_layernorm']
Unsloth: Just some info: will skip parsing ['pre_feedforward_layernorm', 'post_feedforward_layernorm']


In [5]:
# Parameter-Efficient Fine-Tuning (PEFT)
# enable efficient adaptation of large pretrained models to various downstream applications 
# by only fine-tuning a small number of (extra) model parameters instead of all parameters. 
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank*2,               # speeds up training
    use_gradient_checkpointing = "unsloth", # Reduces memory usage
    random_state = 4096,
)

Unsloth 2025.8.9 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


## (2) GRPO chat template

In [6]:
reasoning_start = "<start_breaking_down>"
reasoning_end   = "<end_breaking_down>"
solution_start  = "<SOLUTION>"
solution_end    = "</SOLUTION>"

system_prompt = \
f"""You are given a problem.
Think about the problem and provide your working out.
Place it between {reasoning_start} and {reasoning_end}.
Then, provide your solution between {solution_start}{solution_end}"""
system_prompt

'You are given a problem.\nThink about the problem and provide your working out.\nPlace it between <start_breaking_down> and <end_breaking_down>.\nThen, provide your solution between <SOLUTION></SOLUTION>'

In [7]:
chat_template = \
    "{% if messages[0]['role'] == 'system' %}"\
        "{{ messages[0]['content'] + eos_token }}"\
        "{% set loop_messages = messages[1:] %}"\
    "{% else %}"\
        "{{ '{system_prompt}' + eos_token }}"\
        "{% set loop_messages = messages %}"\
    "{% endif %}"\
    "{% for message in loop_messages %}"\
        "{% if message['role'] == 'user' %}"\
            "{{ message['content'] }}"\
        "{% elif message['role'] == 'assistant' %}"\
            "{{ message['content'] + eos_token }}"\
        "{% endif %}"\
    "{% endfor %}"\
    "{% if add_generation_prompt %}{{ '{reasoning_start}' }}"\
    "{% endif %}"

In [8]:
chat_template = chat_template\
    .replace("'{system_prompt}'",   f"'{system_prompt}'")\
    .replace("'{reasoning_start}'", f"'{reasoning_start}'")
tokenizer.chat_template = chat_template

In [9]:
# Example
tokenizer.apply_chat_template([
    {"role" : "user", "content" : "What is 1+1?"},
    {"role" : "assistant", "content" : f"{reasoning_start}I think it's 2.{reasoning_end}{solution_start}2{solution_end}"},
    {"role" : "user", "content" : "What is 2+2?"},
], tokenize = False, add_generation_prompt = True)

"You are given a problem.\nThink about the problem and provide your working out.\nPlace it between <start_breaking_down> and <end_breaking_down>.\nThen, provide your solution between <SOLUTION></SOLUTION><|endoftext|>What is 1+1?<start_breaking_down>I think it's 2.<end_breaking_down><SOLUTION>2</SOLUTION><|endoftext|>What is 2+2?<start_breaking_down>"

## (3) Dataset for pre fine-tuning

In [10]:
dataset = load_dataset("unsloth/OpenMathReasoning-mini", split = "cot")
dataset = dataset.to_pandas()[
    ["expected_answer", "problem", "generated_solution"]
]

In [11]:
# Keep only the expected_answer is a number
is_number = pd.to_numeric(pd.Series(dataset["expected_answer"]), errors = "coerce").notnull()
dataset = dataset.iloc[np.where(is_number)[0]]

In [12]:
dataset.head()

,expected_answer,problem,generated_solution
0,14,Given $\sqrt{x^2+165}-\sqrt{x^2-52}=7$ and $x$...,"<think>\nOkay, let's see. I need to solve the ..."
6,-2,Find the value of the parameter $a$ for which ...,"<think>\nOkay, so I need to find the value of ..."
9,18,What is the sum of all real numbers $x$ for wh...,"<think>\nOkay, so I need to solve the equation..."
13,2,Evaluate the sum \(\sum_{n=1}^\infty \frac{\ph...,"<think>\nOkay, so I need to evaluate the infin..."
17,30,What is the largest positive integer that divi...,"<think>\nAlright, so I need to find the larges..."


In [13]:
# Adapt the dataset to follow GRPO style formatting:
def format_dataset(x):
    expected_answer = x["expected_answer"]
    problem = x["problem"]
    thoughts = x["generated_solution"]
    thoughts = thoughts.replace("<think>", "").replace("</think>", "")
    thoughts = thoughts.strip()
    
    final_prompt = \
        reasoning_start + thoughts + reasoning_end + \
        solution_start + expected_answer + solution_end
    return [
        {"role" : "system",    "content" : system_prompt},
        {"role" : "user",      "content" : problem},
        {"role" : "assistant", "content" : final_prompt},
    ]

dataset["Messages"] = dataset.apply(format_dataset, axis = 1)

In [14]:
# Test
tokenizer.apply_chat_template(dataset["Messages"][0], tokenize = False)

"You are given a problem.\nThink about the problem and provide your working out.\nPlace it between <start_breaking_down> and <end_breaking_down>.\nThen, provide your solution between <SOLUTION></SOLUTION><|endoftext|>Given $\\sqrt{x^2+165}-\\sqrt{x^2-52}=7$ and $x$ is positive, find all possible values of $x$.<start_breaking_down>Okay, let's see. I need to solve the equation √(x² + 165) - √(x² - 52) = 7, and find all positive values of x. Hmm, radicals can be tricky, but maybe if I can eliminate the square roots by squaring both sides. Let me try that.\n\nFirst, let me write down the equation again to make sure I have it right:\n\n√(x² + 165) - √(x² - 52) = 7.\n\nOkay, so the idea is to isolate one of the radicals and then square both sides. Let me try moving the second radical to the other side:\n\n√(x² + 165) = 7 + √(x² - 52).\n\nNow, if I square both sides, maybe I can get rid of the square roots. Let's do that:\n\n(√(x² + 165))² = (7 + √(x² - 52))².\n\nSimplifying the left side:\n\

In [15]:
# Keep only those with reasonable reasoning traces length
dataset["N"] = dataset["Messages"].apply(lambda x: len(tokenizer.apply_chat_template(x)))
dataset = dataset.loc[dataset["N"] <= max_seq_length/2].copy()
dataset.shape

(58, 5)

In [16]:
# Convert to HF dataset
dataset["text"] = tokenizer.apply_chat_template(dataset["Messages"].values.tolist(), tokenize = False)
dataset = Dataset.from_pandas(dataset)
dataset

Dataset({
    features: ['expected_answer', 'problem', 'generated_solution', 'Messages', 'N', 'text', '__index_level_0__'],
    num_rows: 58
})

## (4) Pre fine-tune the model

In [17]:
# Goal is to make the model following the GRPO formatting
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 1,
        warmup_steps = 5,
        num_train_epochs = 2,
        learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 4096,
        report_to = "none",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/58 [00:00<?, ? examples/s]

In [18]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 58 | Num Epochs = 2 | Total steps = 116
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 1 x 1) = 1
 "-____-"     Trainable parameters = 66,060,288 of 4,088,528,384 (1.62% trained)


Step,Training Loss
5,0.759300
10,0.600700
15,0.450500
20,0.467100
25,0.402000
30,0.399600
35,0.423700
40,0.418100
45,0.448900
50,0.401000


Unsloth: Will smartly offload gradients to save VRAM!


TrainOutput(global_step=116, training_loss=0.3520683407269675, metrics={'train_runtime': 46.716, 'train_samples_per_second': 2.483, 'train_steps_per_second': 2.483, 'total_flos': 2339032341190656.0, 'train_loss': 0.3520683407269675})

In [19]:
# Test
dataset[0]["Messages"][:2]

[{'content': 'You are given a problem.\nThink about the problem and provide your working out.\nPlace it between <start_breaking_down> and <end_breaking_down>.\nThen, provide your solution between <SOLUTION></SOLUTION>',
  'role': 'system'},
 {'content': 'Jenifer has 82 cents in pennies and nickels. Her younger brother mistook all her nickels for dimes and counted the total as $1.47. How many pennies does Jenifer have?',
  'role': 'user'}]

In [20]:
text = tokenizer.apply_chat_template(
    dataset[0]["Messages"][:2],   # keep system(0) & user(1) prompt, without assistant prompt(2)
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
)
text

'You are given a problem.\nThink about the problem and provide your working out.\nPlace it between <start_breaking_down> and <end_breaking_down>.\nThen, provide your solution between <SOLUTION></SOLUTION><|endoftext|>Jenifer has 82 cents in pennies and nickels. Her younger brother mistook all her nickels for dimes and counted the total as $1.47. How many pennies does Jenifer have?<start_breaking_down>'

In [21]:
text

'You are given a problem.\nThink about the problem and provide your working out.\nPlace it between <start_breaking_down> and <end_breaking_down>.\nThen, provide your solution between <SOLUTION></SOLUTION><|endoftext|>Jenifer has 82 cents in pennies and nickels. Her younger brother mistook all her nickels for dimes and counted the total as $1.47. How many pennies does Jenifer have?<start_breaking_down>'

In [22]:
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    temperature = 0,
    max_new_tokens = 1024,
    streamer = TextStreamer(tokenizer, skip_prompt = False),
)

You are given a problem.
Think about the problem and provide your working out.
Place it between <start_breaking_down> and <end_breaking_down>.
Then, provide your solution between <SOLUTION></SOLUTION><|endoftext|>Jenifer has 82 cents in pennies and nickels. Her younger brother mistook all her nickels for dimes and counted the total as $1.47. How many pennies does Jenifer have?<start_breaking_down>Okay, let's see. So the problem is about Jenifer who has 82 cents in pennies and nickels. But her younger brother thought all the nickels were dimes and counted the total as $1.47. We need to find out how many pennies Jenifer has. Hmm, let me break this down.

First, I need to set up some equations. Let's denote the number of pennies as P and the number of nickels as N. Since pennies are worth 1 cent each and nickels 5 cents each, the total value Jenifer has is 1*P + 5*N = 82 cents. That's the first equation.

Now, her brother mistook all the nickels for dimes. Dimes are 10 cents each. So he c

In [23]:
# Clean up
del dataset
torch.cuda.empty_cache()
gc.collect()

0

## (5) Dataset

In [24]:
dataset = load_dataset("open-r1/DAPO-Math-17k-Processed", "en", split = "train")
dataset

README.md: 0.00B [00:00, ?B/s]

en/train-00000-of-00001.parquet:   0%|          | 0.00/5.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/14116 [00:00<?, ? examples/s]

Dataset({
    features: ['prompt', 'solution', 'data_source', 'source_prompt', 'ability', 'reward_model', 'extra_info'],
    num_rows: 14116
})

In [25]:
# Test
print(dataset[100]["prompt"])
print(dataset[100]["solution"])

Three 12 cm $\times$ 12 cm squares are each cut into two pieces $A$ and $B$, as shown in the first figure below, by joining the midpoints of two adjacent sides. These six pieces are then attached to a regular hexagon, as shown in the second figure, so as to fold into a polyhedron. What is the volume (in $\text{cm}^3$) of this polyhedron?
864


In [26]:
dataset = dataset.map(lambda x: {
    "prompt" : [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": x["prompt"]},
    ],
    "answer": x["solution"],
})
dataset[88]

Map:   0%|          | 0/14116 [00:00<?, ? examples/s]

{'prompt': [{'content': 'You are given a problem.\nThink about the problem and provide your working out.\nPlace it between <start_breaking_down> and <end_breaking_down>.\nThen, provide your solution between <SOLUTION></SOLUTION>',
   'role': 'system'},
  {'content': 'Triangle $ABC$ is right-angled at $A$. The circle with center $A$ and radius $AB$ intersects $BC$ and $AC$ internally at points $D$ and $E$ respectively. Given that $BD = 20$ and $DC = 16$, determine $AC^2$.',
   'role': 'user'}],
 'solution': '936',
 'data_source': 'math_dapo',
 'source_prompt': [{'content': 'Solve the following math problem step by step. The last line of your response should be of the form Answer: $Answer (without quotes) where $Answer is the answer to the problem.\n\nTriangle $ABC$ is right-angled at $A$. The circle with center $A$ and radius $AB$ intersects $BC$ and $AC$ internally at points $D$ and $E$ respectively. Given that $BD = 20$ and $DC = 16$, determine $AC^2$.\n\nRemember to put your answer o

In [36]:
tokenized = dataset.map(
    lambda x: {"tokens" : tokenizer.apply_chat_template(x["prompt"], add_generation_prompt = True, tokenize = True)},
    batched = True,
)
print(tokenizer.decode(tokenized[88]["tokens"]))
tokenized = tokenized.map(lambda x: {"L" : len(x["tokens"])})

import numpy as np
maximum_length = int(np.quantile(tokenized["L"], 0.9))
print("Max Length = ", maximum_length)

# Filter only samples smaller than 90% max length
dataset = dataset.select(np.where(np.array(tokenized["L"]) <= maximum_length)[0])
del tokenized

Map:   0%|          | 0/14116 [00:00<?, ? examples/s]

You are given a problem.
Think about the problem and provide your working out.
Place it between <start_breaking_down> and <end_breaking_down>.
Then, provide your solution between <SOLUTION></SOLUTION><|endoftext|>Triangle $ABC$ is right-angled at $A$. The circle with center $A$ and radius $AB$ intersects $BC$ and $AC$ internally at points $D$ and $E$ respectively. Given that $BD = 20$ and $DC = 16$, determine $AC^2$.<start_breaking_down>


Map:   0%|          | 0/14116 [00:00<?, ? examples/s]

Max Length =  204


## (6) Reward function

In [28]:
import re

solution_end_regex = r"</SOLUTION>[\s]{0,}" + \
    "(?:" + re.escape(tokenizer.eos_token) + ")?"
match_format = re.compile(
    rf"{reasoning_end}.*?"\
    rf"{solution_start}(.+?){solution_end_regex}"\
    rf"[\s]{{0,}}$",
    flags = re.MULTILINE | re.DOTALL
)
# Grab the final answer
match_format

re.compile(r'<end_breaking_down>.*?<SOLUTION>(.+?)</SOLUTION>[\s]{0,}(?:<\|endoftext\|>)?[\s]{0,}$',
           re.MULTILINE|re.DOTALL|re.UNICODE)

In [29]:
# Text
match_format.findall(
    "Both conditions are satisfied, confirming that the solution is correct. Therefore, the number of pennies Jenifer has is \(\boxed{17}\).<end_breaking_down><SOLUTION>17</SOLUTION><|endoftext|>"
)

<>:2: SyntaxWarning: invalid escape sequence '\('
<>:2: SyntaxWarning: invalid escape sequence '\('
/tmp/ipykernel_3954/2935232894.py:2: SyntaxWarning: invalid escape sequence '\('
  "Both conditions are satisfied, confirming that the solution is correct. Therefore, the number of pennies Jenifer has is \(\boxed{17}\).<end_breaking_down><SOLUTION>17</SOLUTION><|endoftext|>"


['17']

In [34]:
match_numbers = re.compile(
    solution_start + r".*?[\s]{0,}([-]?[\d\.\,]{1,})",
    flags = re.MULTILINE | re.DOTALL
)
print(match_numbers)

re.compile('<SOLUTION>.*?[\\s]{0,}([-]?[\\d\\.\\,]{1,})', re.MULTILINE|re.DOTALL)


In [35]:
print(match_numbers.findall("<SOLUTION>  0.34  </SOLUTION>"))
print(match_numbers.findall("<SOLUTION>  123,456  </SOLUTION>"))
print(match_numbers.findall("<SOLUTION>  -0.234  </SOLUTION>"))
print(match_numbers.findall("<SOLUTION>17</SOLUTION>"))
print(match_numbers.findall("<SOLUTION>The solution is $20</SOLUTION>"))

['0.34']
['123,456']
['-0.234']
['17']
['20']


In [30]:
def match_format_exactly(completions, **kwargs):
    scores = []
    for completion in completions:
        score = 0
        response = completion[0]["content"]
        if match_format.search(response) is not None: score += 3.0
        scores.append(score)
    return scores

In [31]:
def match_format_approximately(completions, **kwargs):
    scores = []
    for completion in completions:
        score = 0
        response = completion[0]["content"]
        # Count how many keywords are seen - we penalize if too many!
        # If we see 1, then plus some points!
        # No need to reward "reasoning_start" since we always prepend it!
        score += 0.5 if response.count(reasoning_end)   == 1 else -1.0
        score += 0.5 if response.count(solution_start)  == 1 else -1.0
        score += 0.5 if response.count(solution_end)    == 1 else -1.0
        scores.append(score)
    return scores

In [32]:
def check_answer(prompts, completions, answer, **kwargs):
    question = prompts[0][-1]["content"]
    responses = [completion[0]["content"] for completion in completions]

    extracted_responses = [
        guess.group(1)
        if (guess := match_format.search(r)) is not None else None \
        for r in responses
    ]

    scores = []
    for guess, true_answer in zip(extracted_responses, answer):
        score = 0
        if guess is None:
            scores.append(-2.0)
            continue
        # Correct answer gets 5 points!
        if guess == true_answer:
            score += 5.0
        # Match if spaces are seen, but less reward
        elif guess.strip() == true_answer.strip():
            score += 3.5
        else:
            # We also reward it if the answer is close via ratios!
            # Ie if the answer is within some range, reward it!
            try:
                ratio = float(guess) / float(true_answer)
                if   ratio >= 0.9 and ratio <= 1.1: score += 2.0
                elif ratio >= 0.8 and ratio <= 1.2: score += 1.5
                else: score -= 2.5 # Penalize wrong answers
            except:
                score -= 4.5 # Penalize
        scores.append(score)
    return scores

In [38]:
global PRINTED_TIMES
PRINTED_TIMES = 0
global PRINT_EVERY_STEPS
PRINT_EVERY_STEPS = 5

def check_numbers(prompts, completions, answer, **kwargs):
    question = prompts[0][-1]["content"]
    responses = [completion[0]["content"] for completion in completions]

    extracted_responses = [
        guess.group(1)
        if (guess := match_numbers.search(r)) is not None else None \
        for r in responses
    ]

    scores = []
    # Print only every few steps
    global PRINTED_TIMES
    global PRINT_EVERY_STEPS
    if PRINTED_TIMES % PRINT_EVERY_STEPS == 0:
        print(
            '*'*20 + f"Question:\n{question}", f"\nAnswer:\n{answer[0]}", f"\nResponse:\n{responses[0]}", f"\nExtracted:\n{extracted_responses[0]}"
        )
    PRINTED_TIMES += 1

    for guess, true_answer in zip(extracted_responses, answer):
        if guess is None:
            scores.append(-2.5)
            continue
        # Convert to numbers
        try:
            true_answer = float(true_answer.strip())
            # Remove commas like in 123,456
            guess       = float(guess.strip().replace(",", ""))
            scores.append(3.5 if guess == true_answer else -1.5)
        except:
            scores.append(0)
            continue
    return scores